# PySpark - MLP

Durante el desarrollo inicial del modelo, se trabajó con el dataset original  completo y debido a los altos tiempos de procesamiento, que en algunos casos superaban las 12 horas sin completarse. Esta limitación afectaba significativamente la continuidad del trabajo y dificultaba compatibilizar el proyecto con otras responsabilidades académicas y personales, ya que el uso intensivo de recursos ocupaba la mayor parte de la capacidad del equipo.

Por esta razón, se optó inicialmente por un enfoque basado en muestreo para facilitar la experimentación y validación del modelo. Sin embargo, dado que el objetivo del proyecto implica trabajar con grandes volúmenes de datos, se introduce el uso de PySpark como una herramienta que permite escalar el procesamiento y entrenamiento de modelos de manera eficiente, aprovechando capacidades de computación distribuida.

In [1]:
# ==========================================
# 1. Configuración inicial
# ==========================================
import findspark
findspark.init()
 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
import time
 

spark = SparkSession.builder \
    .appName("MLP_CTR") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .config("spark.driver.maxResultSize", "12g") \
    .getOrCreate()
 
spark.sparkContext.setLogLevel("WARN")
 
# ==========================================
# 2. Carga del dataset
# ==========================================


# ==========================================
# 2. Carga del dataset + sample temprano
# ==========================================
DATA_PATH = r"C:\Users\camil\Documents\Estudio\DL\Corte1\Dataset\avazu-ctr-prediction\train.gz"
df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)

#'0.025'
df = df.sample(fraction=0.0525 , seed=42)
df.cache()
print("Columnas:", df.columns)
print("Número de filas tras sample:", df.count())  

Columnas: ['id', 'click', 'hour', 'C1', 'banner_pos', 'site_id', 'site_domain', 'site_category', 'app_id', 'app_domain', 'app_category', 'device_id', 'device_ip', 'device_model', 'device_type', 'device_conn_type', 'C14', 'C15', 'C16', 'C17', 'C18', 'C19', 'C20', 'C21']
Número de filas tras sample: 2123235


In [2]:
# ==========================================
# 3. Feature Engineering
# ==========================================
 
# --- Hora del día y franja horaria ---
df = df.withColumn("hour_day", F.col("hour") % 100)
 
df = df.withColumn(
    "franja",
    F.when((F.col("hour_day") >= 0)  & (F.col("hour_day") < 6),  "Madrugada")
     .when((F.col("hour_day") >= 6)  & (F.col("hour_day") < 12), "Mañana")
     .when((F.col("hour_day") >= 12) & (F.col("hour_day") < 18), "Tarde")
     .otherwise("Noche")
)
 
# --- Mapeos categóricos ---

df = df.withColumn("dct_cat", F.col("device_conn_type").cast("string"))
dt_map_expr = (
    F.when(F.col("device_type") == 0, "0")
     .when(F.col("device_type") == 1, "1")
     .otherwise("Otros")
)
df = df.withColumn("dt_cat", dt_map_expr.cast("string"))
 
bp_map_expr = (
    F.when(F.col("banner_pos") == 6, "6")
     .when(F.col("banner_pos") == 7, "7")
     .otherwise("Otros")
)
df = df.withColumn("bp_cat", bp_map_expr.cast("string"))
 
# --- Conteos por grupo (count encoding) ---

count_cols = ['device_ip', 'device_id', 'device_model', 'app_id', 'site_id']
for c in count_cols:
    w = Window.partitionBy(c)
    df = df.withColumn(f"{c}_count", F.count(c).over(w))
 
# --- Features derivadas (ratio encoding) ---

df = df.withColumn("app_per_device",
    F.col("app_id_count") / (F.col("device_id_count") + 1))
df = df.withColumn("site_per_device",
    F.col("site_id_count") / (F.col("device_id_count") + 1))

En esta etapa se realiza el proceso de feature engineering (De manera análoga  a la sección de Scikit-learn)con el objetivo de enriquecer la información disponible para el modelo. En primer lugar, se extrae la hora del día a partir de la variable original y se construye una variable categórica de franja horaria, permitiendo capturar patrones de comportamiento según momentos del día.

Posteriormente, se llevan a cabo transformaciones sobre variables categóricas mediante agrupaciones y simplificaciones, reduciendo la cardinalidad y facilitando su posterior codificación. 

Adicionalmente, se implementa count encoding sobre variables de alta cardinalidad, generando nuevas características basadas en la frecuencia de aparición de identificadores como dispositivos, aplicaciones y sitios. Finalmente, se crean variables derivadas tipo ratio, que capturan relaciones entre conteos (por ejemplo, número de aplicaciones por dispositivo), aportando información adicional relevante para el modelo.

In [3]:
# ==========================================
# 4. Definir variables
# ==========================================
target_col = "click"
 
# High cardinality → StringIndexer (embedding implícito por índice)
high_card_cols = ['C14', 'C17', 'C19', 'C20', 'C21']
 
# Low cardinality → OneHotEncoder
low_card_cols = ['app_category', 'site_category', 'bp_cat', 'dct_cat',
                 'dt_cat', 'C1', 'C18', 'C15', 'C16', 'franja']
 
# Numéricas (incluyendo count features y ratios)
num_vars = ["hour_day", "device_ip_count", "device_id_count",
                 "app_id_count", "site_id_count", "app_per_device", "site_per_device"
]

In [4]:
# ==========================================
# 5. Codificación y Pipeline
# ==========================================
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in high_card_cols
]
 
low_indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_str_idx", handleInvalid="keep")
    for c in low_card_cols
]
 
ohe = OneHotEncoder(
    inputCols=[f"{c}_str_idx" for c in low_card_cols],
    outputCols=[f"{c}_ohe" for c in low_card_cols],
    handleInvalid="keep"
)
 
assembler_inputs = (
    [f"{c}_idx"  for c in high_card_cols] +
    [f"{c}_ohe"  for c in low_card_cols] +
    num_vars
)
assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features_unscaled",
    handleInvalid="keep"   
)
 
scaler = StandardScaler(
    inputCol="features_unscaled",
    outputCol="features",
    withStd=True,
    withMean=False  
)
 
pipeline_prep = Pipeline(stages=indexers + low_indexers + [ohe, assembler, scaler])
 
df_prepared = pipeline_prep.fit(df).transform(df)
df_prepared = df_prepared.select("features", F.col(target_col).cast("double"))
df_prepared.cache()  

DataFrame[features: vector, click: double]

En esta etapa se construye el pipeline de transformación de variables utilizando las herramientas de PySpark ML. Primero, se aplican transformaciones de indexación a las variables categóricas de alta y baja cardinalidad mediante `StringIndexer`, permitiendo convertirlas en representaciones numéricas.

Para las variables de baja cardinalidad, se aplica posteriormente codificación One-Hot, generando vectores binarios que preservan la información categórica sin introducir relaciones ordinales artificiales.

Una vez transformadas las variables, se integran todas las características —numéricas, categóricas indexadas y codificadas— en un único vector mediante `VectorAssembler`. Finalmente, se realiza un escalado de las variables con `StandardScaler`, lo que mejora la estabilidad y el desempeño del modelo neuronal.

Todo este flujo se encapsula en un pipeline, lo que permite una transformación eficiente, reproducible y escalable de los datos dentro del entorno distribuido de PySpark.

In [5]:
# ==========================================
# 6. División estratificada train/test
# ==========================================
def stratified_split(df, label_col, train_ratio=0.8, seed=42):
    pos = df.filter(F.col(label_col) == 1)
    neg = df.filter(F.col(label_col) == 0)
    pos_train, pos_test = pos.randomSplit([train_ratio, 1 - train_ratio], seed=seed)
    neg_train, neg_test = neg.randomSplit([train_ratio, 1 - train_ratio], seed=seed)
    return pos_train.union(neg_train), pos_test.union(neg_test)
 
train, test = stratified_split(df_prepared, target_col)
 
print(f"Train: {train.count()} filas | Test: {test.count()} filas")
print("Distribución train:", train.groupBy(target_col).count().show())
print("Distribución test:",  test.groupBy(target_col).count().show())

Train: 1698304 filas | Test: 424931 filas
+-----+-------+
|click|  count|
+-----+-------+
|  1.0| 288494|
|  0.0|1409810|
+-----+-------+

Distribución train: None
+-----+------+
|click| count|
+-----+------+
|  1.0| 72364|
|  0.0|352567|
+-----+------+

Distribución test: None


Se implementa una estrategia de división estratificada para separar los datos en conjuntos de entrenamiento y prueba. Para ello, se divide el dataset por clases (click y no click), aplicando posteriormente un muestreo aleatorio independiente en cada grupo.

Este enfoque permite mantener proporciones similares de cada clase en ambos conjuntos, lo cual es fundamental para obtener una evaluación más representativa del desempeño del modelo. Finalmente, se validan las distribuciones resultantes para asegurar la correcta preservación del balance de clases.

In [6]:
# ==========================================
# 6b. Undersampling para balancear clases
# ==========================================
count_pos = train.filter(F.col(target_col) == 1.0).count()
count_neg = train.filter(F.col(target_col) == 0.0).count()
total_before = count_pos + count_neg

print("========== ANTES DEL UNDERSAMPLING ==========")
print(f"Total filas train : {total_before}")
print(f"Clase 1 (click)   : {count_pos} ({count_pos/total_before*100:.2f}%)")
print(f"Clase 0 (no click): {count_neg} ({count_neg/total_before*100:.2f}%)")

# Ratio para reducir la clase mayoritaria al tamaño de la minoritaria
ratio = count_pos / count_neg

pos_train = train.filter(F.col(target_col) == 1.0)
neg_train = train.filter(F.col(target_col) == 0.0).sample(fraction=ratio, seed=42)

train_balanced = pos_train.union(neg_train).orderBy(F.rand(seed=42))
train_balanced.cache()

count_pos_b = train_balanced.filter(F.col(target_col) == 1.0).count()
count_neg_b = train_balanced.filter(F.col(target_col) == 0.0).count()
total_after = count_pos_b + count_neg_b

print("\n========== DESPUÉS DEL UNDERSAMPLING ==========")
print(f"Total filas train : {total_after}")
print(f"Clase 1 (click)   : {count_pos_b} ({count_pos_b/total_after*100:.2f}%)")
print(f"Clase 0 (no click): {count_neg_b} ({count_neg_b/total_after*100:.2f}%)")

========== ANTES DEL UNDERSAMPLING ==========
Total filas train : 1698304
Clase 1 (click)   : 288494 (16.99%)
Clase 0 (no click): 1409810 (83.01%)

========== DESPUÉS DEL UNDERSAMPLING ==========
Total filas train : 576824
Clase 1 (click)   : 288494 (50.01%)
Clase 0 (no click): 288330 (49.99%)


A diferencia de la implementación en scikit-learn, donde el desbalance de clases se abordó mediante técnicas como el uso de pesos o métodos de sobremuestreo como SMOTE (que generan datos sintéticos), en esta etapa se opta por una estrategia de *undersampling* sobre la clase mayoritaria.

La elección de *undersampling* se justifica principalmente por el gran volumen de datos disponible, lo que permite reducir la clase dominante sin perder de forma significativa la representatividad de la información. Además, esta técnica resulta más eficiente computacionalmente en entornos distribuidos como PySpark, evitando el costo adicional de generar nuevas observaciones sintéticas.

En concreto, se reduce aleatoriamente el número de instancias de la clase "no click" hasta igualar la cantidad de la clase minoritaria ("click"), obteniendo así un conjunto de entrenamiento balanceado. Esto permite mitigar el sesgo hacia la clase mayoritaria y mejora la capacidad del modelo para detectar correctamente la clase positiva.

In [7]:
# ==========================================
# 7. Modelo MLP + CrossValidator
# ==========================================
num_features = len(train_balanced.select("features").first()[0])
print(f"Número de features: {num_features}")
 
mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=target_col,
    seed=42,
    blockSize=128  
)
 
paramGrid = (
    ParamGridBuilder()
    .addGrid(mlp.layers, [
        [num_features, 10, 2],
        [num_features, 50, 2],
        [num_features, 70, 2],
    ])
    .addGrid(mlp.stepSize, [0.1, 0.01])
    .addGrid(mlp.maxIter,  [500, 700])
    .build()
)
 
evaluator_auc = BinaryClassificationEvaluator(
    labelCol=target_col,
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
 
cv = CrossValidator(
    estimator=mlp,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator_auc,
    numFolds=3,
    parallelism=10   
)

Número de features: 113


Se define y entrena un modelo de red neuronal tipo Multilayer Perceptron (MLP) utilizando PySpark. Se especifica la arquitectura del modelo en función del número de variables de entrada y se exploran diferentes configuraciones de capas ocultas, tasa de aprendizaje y número de iteraciones.

Para optimizar el desempeño del modelo, se emplea `CrossValidator`, permitiendo evaluar múltiples combinaciones de hiperparámetros mediante validación cruzada. Como métrica principal se utiliza el área bajo la curva ROC (AUC), adecuada para problemas de clasificación binaria desbalanceada. Además, se aprovecha el paralelismo de Spark para acelerar el proceso de búsqueda de hiperparámetros.

In [8]:
# ==========================================
# 8. Entrenamiento
# ==========================================
start_time = time.time()
cv_model = cv.fit(train_balanced)
train_time = time.time() - start_time
print(f"Tiempo de entrenamiento: {train_time:.2f} s")

Tiempo de entrenamiento: 19506.20 s


El ouput indica un total un total de 5 horas con 25 minutos de duramiento del entrenamiento, lo cual muestra la robustez del uso de pyspark

In [9]:
# ==========================================
# 9. Predicción
# ==========================================
start_pred = time.time()
predictions = cv_model.transform(test)
pred_time = time.time() - start_pred
print(f"Tiempo de predicción: {pred_time:.2f} s")
print(f"Tiempo total (entreno + predicción): {train_time + pred_time:.2f} s")

Tiempo de predicción: 0.02 s
Tiempo total (entreno + predicción): 19506.22 s


In [10]:
# ==========================================
# 10. Evaluación completa
# CORRECCIÓN: el código original solo calculaba F1 y AUC.
# Los requisitos piden también Precisión, Recall y Matriz de Confusión.
# ==========================================
 
# --- AUC ROC ---
auc = evaluator_auc.evaluate(predictions)
 
# --- F1, Precisión, Recall ---
def mc_eval(metric):
    return MulticlassClassificationEvaluator(
        labelCol=target_col, predictionCol="prediction", metricName=metric
    ).evaluate(predictions)
 
f1        = mc_eval("f1")
precision = mc_eval("weightedPrecision")
recall    = mc_eval("weightedRecall")
accuracy  = mc_eval("accuracy")
 
print("\n========== MÉTRICAS ==========")
print(f"AUC-ROC   : {auc:.4f}")
print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"Precisión : {precision:.4f}")
print(f"Recall    : {recall:.4f}")


========== MÉTRICAS ==========
AUC-ROC   : 0.6985
Accuracy  : 0.6250
F1-Score  : 0.6714
Precisión : 0.7931
Recall    : 0.6250


Los resultados obtenidos muestran un desempeño moderado del modelo. El valor de **AUC-ROC** (0.6985) indica una capacidad aceptable para discriminar entre las clases, aunque aún existe margen de mejora.

En términos de clasificación, el modelo presenta una **precisión** alta (0.79), lo que significa que cuando predice un click, suele acertar en la mayoría de los casos. Sin embargo, el **recall** (0.62) es más bajo, lo que indica que el modelo no logra identificar todos los clics reales, dejando escapar una parte de ellos.

El **F1-score** (0.67) refleja un balance razonable entre precisión y recall, sugiriendo que el modelo mantiene un compromiso adecuado entre detectar la clase positiva y evitar falsos positivos. Por su parte, la accuracy (0.62) es consistente con este comportamiento general.

En conjunto, el modelo tiende a ser más conservador, priorizando la precisión sobre la cobertura total de la clase positiva. Esto puede ser útil en escenarios donde se busca reducir falsos positivos, aunque podría mejorarse si el objetivo es maximizar la detección de clics.

In [11]:
# --- Matriz de confusión ---
print("\n========== MATRIZ DE CONFUSIÓN ==========")
conf_matrix = (
    predictions
    .groupBy(F.col(target_col).alias("Real"),
             F.col("prediction").alias("Predicho"))
    .count()
    .orderBy("Real", "Predicho")
)
conf_matrix.show()


========== MATRIZ DE CONFUSIÓN ==========
+----+--------+------+
|Real|Predicho| count|
+----+--------+------+
| 0.0|     0.0|216863|
| 0.0|     1.0|135704|
| 1.0|     0.0| 23659|
| 1.0|     1.0| 48705|
+----+--------+------+



La matriz de confusión evidencia que el modelo clasifica correctamente una gran cantidad de casos de la clase negativa (no click), con 216,863 aciertos. Sin embargo, también presenta un número considerable de falsos positivos (135,704), lo que indica que en varias ocasiones predice clics que no ocurren.

En cuanto a la clase positiva (click), el modelo logra identificar 48,705 casos correctamente, pero deja de detectar 23,659 (falsos negativos), lo que refleja una capacidad moderada de detección.

En conjunto, el modelo muestra un comportamiento consistente con las métricas anteriores: buena precisión en la predicción de clics, pero con un trade-off importante en términos de falsos positivos y una cobertura incompleta de la clase positiva.

In [12]:
# --- Mejor modelo encontrado ---
best_model = cv_model.bestModel
print("\n========== MEJOR MODELO ==========")
print(f"Capas      : {best_model.getLayers()}")
print(f"stepSize   : {best_model.getStepSize()}")
print(f"maxIter    : {best_model.getMaxIter()}")
 
spark.stop()


========== MEJOR MODELO ==========
Capas      : [113, 50, 2]
stepSize   : 0.1
maxIter    : 700
